In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS1_DRIVE_ROOT = DRIVE_ROOT / 'OASIS-1'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
LOCAL_OASIS_STAGE = Path('/content/oasis_stage/OASIS')
RUN_NAME = 'oasis_colab_full_v3_auroc_monitor'

if not OASIS1_DRIVE_ROOT.exists():
    raise FileNotFoundError(f'OASIS-1 folder not found: {OASIS1_DRIVE_ROOT}')

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
print('drive_root =', DRIVE_ROOT)
print('oasis1_drive_root =', OASIS1_DRIVE_ROOT)
print('runtime_root =', RUNTIME_ROOT)
print('local_oasis_stage =', LOCAL_OASIS_STAGE)
print('run_name =', RUN_NAME)


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/Cerebrasense-')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', 'https://github.com/Billrichard209/Cerebrasense-.git'],
    cwd='/content',
    check=True,
)

if not BACKEND_ROOT.exists():
    raise FileNotFoundError(f'Expected backend after clone: {BACKEND_ROOT}')

print('repo_root =', REPO_ROOT)
print('backend_root =', BACKEND_ROOT)


In [ ]:
import subprocess
import sys

requirements = BACKEND_ROOT / 'requirements-colab.txt'
if not requirements.exists():
    raise FileNotFoundError(f'Missing requirements file: {requirements}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)
print('dependencies ready =', requirements)


In [ ]:
EPOCHS = 28
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 1
NUM_WORKERS = 0
IMAGE_SIZE = (64, 64, 64)
SEED = 42
SPLIT_SEED = 42
WEIGHTED_SAMPLING = False
EARLY_STOPPING_MONITOR = 'val_auroc'
EARLY_STOPPING_MODE = 'max'
EARLY_STOPPING_PATIENCE = 6
EARLY_STOPPING_MIN_DELTA = 0.0
EVALUATE_SPLITS = ('val', 'test')
STAGE_OASIS_TO_LOCAL = True
FORCE_RESTAGE = False
RESUME_IF_AVAILABLE = True
CALIBRATE_THRESHOLD = True
PROMOTE_BASELINE = True
SELECTION_METRIC = 'f1'
THRESHOLD_STEP = 0.01

print('epochs =', EPOCHS)
print('batch_size =', BATCH_SIZE)
print('gradient_accumulation_steps =', GRADIENT_ACCUMULATION_STEPS)
print('evaluate_splits =', EVALUATE_SPLITS)
print('stage_oasis_to_local =', STAGE_OASIS_TO_LOCAL)
print('resume_if_available =', RESUME_IF_AVAILABLE)
print('calibrate_threshold =', CALIBRATE_THRESHOLD)
print('promote_baseline =', PROMOTE_BASELINE)


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/train_oasis_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--oasis-source-dir', str(OASIS1_DRIVE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--image-size', *(str(value) for value in IMAGE_SIZE),
    '--seed', str(SEED),
    '--split-seed', str(SPLIT_SEED),
    '--early-stopping-monitor', EARLY_STOPPING_MONITOR,
    '--early-stopping-mode', EARLY_STOPPING_MODE,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--early-stopping-min-delta', str(EARLY_STOPPING_MIN_DELTA),
    '--selection-metric', SELECTION_METRIC,
    '--threshold-step', str(THRESHOLD_STEP),
    '--evaluate-splits', *EVALUATE_SPLITS,
    '--local-oasis-root', str(LOCAL_OASIS_STAGE),
]

cmd.append('--weighted-sampling' if WEIGHTED_SAMPLING else '--no-weighted-sampling')
cmd.append('--stage-oasis-to-local' if STAGE_OASIS_TO_LOCAL else '--no-stage-oasis-to-local')
cmd.append('--resume-if-available' if RESUME_IF_AVAILABLE else '--no-resume-if-available')
cmd.append('--calibrate-threshold' if CALIBRATE_THRESHOLD else '--no-calibrate-threshold')
cmd.append('--promote' if PROMOTE_BASELINE else '--no-promote')
if FORCE_RESTAGE:
    cmd.append('--force-restage')

print('RUNNING:', ' '.join(cmd))
subprocess.run(cmd, cwd=BACKEND_ROOT, check=True)


In [ ]:
import json
from pathlib import Path

RUNTIME_DATA_ROOT = RUNTIME_ROOT / 'data'
RUNTIME_OUTPUTS_ROOT = RUNTIME_ROOT / 'outputs'
RUN_ROOT = RUNTIME_OUTPUTS_ROOT / 'runs' / 'oasis' / RUN_NAME
SUMMARY_PATH = RUN_ROOT / 'reports' / 'colab_run_summary.json'
REGISTRY_PATH = RUNTIME_OUTPUTS_ROOT / 'model_registry' / 'oasis_current_baseline.json'

print('runtime_data_root =', RUNTIME_DATA_ROOT)
print('runtime_outputs_root =', RUNTIME_OUTPUTS_ROOT)
print('run_root =', RUN_ROOT)
print('summary_path =', SUMMARY_PATH)
print('registry_path =', REGISTRY_PATH)

if SUMMARY_PATH.exists():
    print(json.dumps(json.loads(SUMMARY_PATH.read_text(encoding='utf-8')), indent=2))
else:
    print('summary file not found yet')

if REGISTRY_PATH.exists():
    registry_payload = json.loads(REGISTRY_PATH.read_text(encoding='utf-8'))
    print('promoted_run_name =', registry_payload.get('run_name'))
    print('recommended_threshold =', registry_payload.get('recommended_threshold'))
else:
    print('registry file not found yet')


In [ ]:
# Optional: run one sample inference from the persisted v3 baseline.
import json
import os
import pandas as pd
from pathlib import Path
import sys

RUNTIME_DATA_ROOT = RUNTIME_ROOT / 'data'
RUNTIME_OUTPUTS_ROOT = RUNTIME_ROOT / 'outputs'
manifest_path = RUNTIME_DATA_ROOT / 'interim' / 'oasis1_manifest.csv'
registry_path = RUNTIME_OUTPUTS_ROOT / 'model_registry' / 'oasis_current_baseline.json'

if not manifest_path.exists():
    raise FileNotFoundError(f'Manifest missing: {manifest_path}')
if not registry_path.exists():
    raise FileNotFoundError(f'Registry missing: {registry_path}')

os.environ['ALZ_DATA_ROOT'] = str(RUNTIME_DATA_ROOT)
os.environ['ALZ_OUTPUTS_ROOT'] = str(RUNTIME_OUTPUTS_ROOT)
os.environ['ALZ_OASIS_SOURCE_DIR'] = str(LOCAL_OASIS_STAGE if LOCAL_OASIS_STAGE.exists() else (OASIS1_DRIVE_ROOT / 'OASIS' if (OASIS1_DRIVE_ROOT / 'OASIS').exists() else OASIS1_DRIVE_ROOT))
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

import torch
from src.evaluation.calibration import ConfidenceBandConfig
from src.inference.pipeline import PredictScanOptions, predict_scan

manifest = pd.read_csv(manifest_path)
sample_row = manifest.iloc[0]
registry_payload = json.loads(registry_path.read_text(encoding='utf-8'))
checkpoint_path = Path(registry_payload['checkpoint_path'])
preprocessing_config_path = Path(registry_payload['preprocessing_config_path'])
model_config_path = Path(registry_payload['model_config_path'])
confidence_config = ConfidenceBandConfig(
    temperature=float(registry_payload.get('temperature_scaling', {}).get('temperature', 1.0)),
    high_confidence_min=float(registry_payload.get('confidence_policy', {}).get('high_confidence_min', 0.85)),
    medium_confidence_min=float(registry_payload.get('confidence_policy', {}).get('medium_confidence_min', 0.65)),
    high_entropy_max=float(registry_payload.get('confidence_policy', {}).get('high_entropy_max', 0.35)),
    medium_entropy_max=float(registry_payload.get('confidence_policy', {}).get('medium_entropy_max', 0.90)),
)
resolved_device = 'cuda' if torch.cuda.is_available() else 'cpu'
payload = predict_scan(
    str(Path(sample_row['image'])),
    str(checkpoint_path),
    str(preprocessing_config_path),
    options=PredictScanOptions(
        output_name='oasis_single_scan_demo',
        threshold=float(registry_payload.get('recommended_threshold', 0.5)),
        device=resolved_device,
        model_config_path=model_config_path,
        subject_id=str(sample_row['subject_id']),
        session_id=None,
        confidence_config=confidence_config,
    ),
)

print('sample_scan_path =', sample_row['image'])
print(json.dumps(payload, indent=2))
